# Assignment 5: Build and Evaluate Classification Models  
## Multi-Class Prediction of Obesity Risk Kaggle Notebook

**Todd DeLozier, MBA, Data+**  
**Predictive Analytics, DDS-855**  
**National University**

In this notebook, I work through the Kaggle portion of Assignment 5 using the attached `train.csv` and `test.csv` files stored in the same local folder as this notebook. I document my exploratory analysis, preprocessing decisions, model construction, assumption review, internal validation results, and Kaggle submission preparation for four required classifiers: regularized multinomial logistic regression, linear discriminant analysis, naive Bayes, and support vector machine.

This notebook is designed to run locally as long as `train.csv` and `test.csv` are saved in the same folder.

## Local File Requirement

This notebook expects the following files in the current working directory:

- `train.csv`
- `test.csv`

The Kaggle competition target is `NObeyesdad`, and the row identifier is `id`.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

In [ ]:
project_dir = Path.cwd()
train_path = project_dir / "train.csv"
test_path = project_dir / "test.csv"

assert train_path.exists(), f"Missing file: {train_path}"
assert test_path.exists(), f"Missing file: {test_path}"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

target_col = "NObeyesdad"
id_col = "id"

print("Project directory:", project_dir)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTraining columns:")
print(train_df.columns.tolist())

## 1. Initial Inspection

I begin by confirming the structure of the training and test files, checking variable types, and verifying whether the data contain missing values. This is important because all four classifiers require a consistent preprocessing pipeline, and LDA and logistic regression are especially sensitive to data quality and redundant encoding issues.

In [ ]:
display(train_df.head())
display(test_df.head())

summary_df = pd.DataFrame({
    "train_dtype": train_df.dtypes.astype(str),
    "train_missing": train_df.isna().sum(),
    "test_dtype": test_df.reindex(columns=train_df.drop(columns=[target_col]).columns).dtypes.astype(str),
    "test_missing": test_df.isna().sum().reindex(train_df.drop(columns=[target_col]).columns)
})
display(summary_df)
print("\nTotal missing values in train:", int(train_df.isna().sum().sum()))
print("Total missing values in test:", int(test_df.isna().sum().sum()))

## 2. Exploratory Analysis

Because this is a multi-class obesity classification problem, I want to understand class balance, numerical distributions, and basic relationships among the continuous predictors. I do not expect perfect normality because the data include synthetic and behavioral features, but these summaries still help me evaluate whether the assumptions behind LDA, naive Bayes, and logistic regression are reasonable approximations.

In [ ]:
numeric_cols = [c for c in train_df.columns if c not in [target_col, id_col] and pd.api.types.is_numeric_dtype(train_df[c])]
categorical_cols = [c for c in train_df.columns if c not in numeric_cols + [target_col, id_col]]

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

display(train_df[numeric_cols].describe().T)

target_counts = train_df[target_col].value_counts().sort_values(ascending=False)
display(target_counts.to_frame("count"))

plt.figure(figsize=(10, 5))
target_counts.plot(kind="bar")
plt.title("Target Class Distribution in Training Data")
plt.xlabel("Obesity Class")
plt.ylabel("Count")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for ax, col in zip(axes, numeric_cols):
    ax.hist(train_df[col], bins=30)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
plt.imshow(train_df[numeric_cols].corr(), cmap="Blues", aspect="auto")
plt.colorbar()
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        plt.text(j, i, f"{train_df[numeric_cols].corr().iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation Matrix for Numeric Predictors")
plt.tight_layout()
plt.show()

The categorical fields are small enough that one-hot encoding is appropriate. I inspect them here to make sure the levels in the test set are compatible with the training set and to confirm that no category dominates the data so completely that the resulting indicators become degenerate.

In [ ]:
for col in categorical_cols:
    print(f"\nValue counts for {col}")
    display(train_df[col].value_counts(dropna=False).to_frame("count"))

## 3. Preprocessing Strategy

I separate the target variable from the predictors, keep the `id` column only for submission output, and build a reusable preprocessing pipeline. Numeric features are median-imputed and standardized. Categorical features are mode-imputed and one-hot encoded with the first level dropped to reduce unnecessary redundancy.

I use a compatibility wrapper for `OneHotEncoder` so that this notebook can run across multiple scikit-learn versions.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, drop="first")

X = train_df.drop(columns=[target_col])
y = train_df[target_col]

X_model = X.drop(columns=[id_col]).copy()
X_test_model = test_df.drop(columns=[id_col]).copy()

numeric_features = [c for c in X_model.columns if pd.api.types.is_numeric_dtype(X_model[c])]
categorical_features = [c for c in X_model.columns if c not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", make_one_hot_encoder())
            ]),
            categorical_features
        )
    ],
    remainder="drop"
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training partition:", X_train.shape, y_train.shape)
print("Validation partition:", X_valid.shape, y_valid.shape)

## 4. Required Classification Models

The assignment requires four models. I implement them as follows:

Regularized multinomial logistic regression is my regression-based classifier and also satisfies the rubric's expectation that I apply regularization. Linear discriminant analysis provides a parametric linear decision boundary. Gaussian naive Bayes gives me a simple probabilistic benchmark. The support vector machine allows a flexible non-linear boundary through a polynomial kernel while remaining fast enough to train and score locally on the attached files.

I evaluate all four models on the same holdout partition so that the comparisons are consistent.

In [ ]:
models = {
    "Regularized Multinomial Logistic Regression": LogisticRegression(
        solver="lbfgs",
        C=1.0,
        max_iter=3000,
        random_state=42
    ),
    "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(
        kernel="poly",
        degree=3,
        C=1.0,
        gamma="scale"
    )
}

def evaluate_model(name, estimator, X_train, y_train, X_valid, y_valid):
    pipeline = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", clone(estimator))
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_valid)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_valid,
        preds,
        average="macro",
        zero_division=0
    )

    result = {
        "Model": name,
        "Accuracy": accuracy_score(y_valid, preds),
        "Macro_Precision": precision,
        "Macro_Recall": recall,
        "Macro_F1": f1
    }
    return pipeline, preds, result

fitted_models = {}
validation_predictions = {}
results = []

for name, estimator in models.items():
    fitted_model, preds, result = evaluate_model(
        name, estimator, X_train, y_train, X_valid, y_valid
    )
    fitted_models[name] = fitted_model
    validation_predictions[name] = preds
    results.append(result)

results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
display(results_df)

## 5. Validation Results and Interpretation

The model comparison table allows me to identify which method generalizes best on my internal holdout set before I upload predictions to Kaggle. I pay attention to both accuracy and macro F1 because this is a seven-class classification problem and I want performance to remain balanced across obesity levels rather than being driven by the largest classes only.

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = fitted_models[best_model_name]

print("Best validation model:", best_model_name)
print("\nDetailed classification report for the best model:\n")
print(classification_report(y_valid, validation_predictions[best_model_name]))

The confusion matrices help me identify which obesity levels are easiest to classify and which levels the models tend to confuse with neighboring classes. In problems like this one, the middle categories often overlap more than the extremes, so I expect the cleanest separation near underweight and severe obesity.

In [ ]:
for name, preds in validation_predictions.items():
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(
        y_valid,
        preds,
        xticks_rotation=35,
        ax=ax,
        colorbar=False
    )
    ax.set_title(f"Confusion Matrix: {name}")
    plt.tight_layout()
    plt.show()

## 6. Assumption Review

I review the core assumptions behind each model rather than treating the algorithms as black boxes.

For regularized multinomial logistic regression, the main concerns are independent observations, a stable relationship between predictors and class log-odds, and manageable multicollinearity. The data contain no missing values, the sample size is large, and regularization helps control coefficient instability. At the same time, the relationship between obesity classes and behavior-related predictors is unlikely to be perfectly linear in the logit, so the model should be interpreted as a useful approximation rather than a perfect parametric description.

For linear discriminant analysis, the strongest assumptions are approximate multivariate normality within class and comparable covariance structure across classes. Because the dataset mixes continuous and one-hot encoded categorical variables, these assumptions are not strictly satisfied. I still include LDA because it remains a classic benchmark, but I interpret its performance with caution.

For naive Bayes, the key assumption is conditional independence among predictors given the class. That assumption is almost certainly violated here because weight, height, dietary behavior, transportation mode, and activity level are not independent in real life. I therefore expect naive Bayes to run quickly but to be less competitive than the other methods.

For the support vector machine, there is no normality assumption, but the method is sensitive to scaling and hyperparameter choices. Standardizing numeric variables and tuning the penalty parameter help the model learn a more stable decision boundary.

The numeric diagnostics below help me assess whether severe collinearity or extreme imbalance is likely to distort the comparison.

In [ ]:
print("Class proportions in the training data:")
display((train_df[target_col].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

print("\nCorrelation matrix for numeric predictors:")
display(train_df[numeric_cols].corr().round(3))

cov_summary = []
for cls, group in train_df.groupby(target_col):
    cov_matrix = group[numeric_cols].cov()
    cov_summary.append({
        "Class": cls,
        "Covariance_Trace": np.trace(cov_matrix.values),
        "Covariance_Determinant": np.linalg.det(cov_matrix.values)
    })

cov_summary_df = pd.DataFrame(cov_summary).sort_values("Class").reset_index(drop=True)
display(cov_summary_df)

## 7. Model-Specific Interpretation

To interpret the regularized logistic regression model, I extract the transformed feature names and inspect the largest absolute coefficients for each class. This does not replace a full causal interpretation, but it does show which encoded predictors exert the strongest directional pull on the class decision rule.

In [ ]:
logit_pipe = fitted_models["Regularized Multinomial Logistic Regression"]
logit_model = logit_pipe.named_steps["model"]
logit_features = logit_pipe.named_steps["prep"].get_feature_names_out()

top_rows = []
for class_name, coef_row in zip(logit_model.classes_, logit_model.coef_):
    top_idx = np.argsort(np.abs(coef_row))[-10:][::-1]
    for rank, idx in enumerate(top_idx, start=1):
        top_rows.append({
            "Class": class_name,
            "Rank": rank,
            "Feature": logit_features[idx],
            "Coefficient": coef_row[idx]
        })

top_coef_df = pd.DataFrame(top_rows)
display(top_coef_df)

The support vector machine is less transparent than logistic regression, but I can still inspect how many support vectors define the final margin. A large number of support vectors usually indicates that the boundary is complex and that many observations contribute to the separating surface.

In [ ]:
svm_pipe = fitted_models["Support Vector Machine"]
svm_model = svm_pipe.named_steps["model"]

support_vector_summary = pd.DataFrame({
    "Class": svm_model.classes_,
    "Support_Vectors": svm_model.n_support_
})
display(support_vector_summary)
print("Total support vectors:", int(svm_model.n_support_.sum()))

For naive Bayes, the most interpretable parameters are the class priors. These priors should closely reflect the observed training distribution because the classes are moderately balanced, although not perfectly uniform.

In [ ]:
nb_pipe = fitted_models["Naive Bayes"]
nb_model = nb_pipe.named_steps["model"]

class_prior_df = pd.DataFrame({
    "Class": nb_model.classes_,
    "Class_Prior": nb_model.class_prior_
}).sort_values("Class").reset_index(drop=True)

display(class_prior_df)

## 8. Full Training and Kaggle Submission Files

Once I finish internal validation, I retrain each classifier on the full training set and generate four separate submission files for Kaggle. Each file contains the required `id` and `NObeyesdad` columns and can be uploaded independently so that I can document the public leaderboard results for all four required methods.

In [ ]:
submission_files = {}

for name, estimator in models.items():
    full_pipeline = Pipeline([
        ("prep", clone(preprocessor)),
        ("model", clone(estimator))
    ])
    full_pipeline.fit(X_model, y)
    test_preds = full_pipeline.predict(X_test_model)

    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("(", "")
        .replace(")", "")
    )
    file_name = f"submission_{safe_name}.csv"

    submission_df = pd.DataFrame({
        "id": test_df[id_col],
        "NObeyesdad": test_preds
    })
    submission_df.to_csv(project_dir / file_name, index=False)
    submission_files[name] = file_name

submission_files_df = pd.DataFrame(
    {"Model": list(submission_files.keys()), "Submission_File": list(submission_files.values())}
).sort_values("Model").reset_index(drop=True)

display(submission_files_df)

In [ ]:
for model_name, file_name in submission_files.items():
    print(f"\nPreview of {file_name}")
    display(pd.read_csv(project_dir / file_name).head())

## 9. Kaggle Evidence Placeholder

After uploading the four submission files to Kaggle, I will paste a screenshot of my submissions and public scores directly below this section before final submission of the assignment.

**Insert Kaggle screenshot here.**

## 10. GitHub Repository Placeholder

The rubric also requires that I post my code to GitHub. I will paste my repository URL below after uploading the notebook and any supporting files.

**Insert GitHub repository URL here.**

## 11. Closing Interpretation

This notebook gives me a complete and reproducible workflow for the Kaggle portion of the assignment. I began by auditing the data, then encoded the mixed predictor set through a common preprocessing pipeline, and finally compared four required classification methods on a holdout validation sample. In general, I expect the support vector machine and the regularized multinomial logistic regression to perform best because the obesity labels are influenced by both continuous body measures and non-linear interactions among behavioral features. LDA is informative as a classical benchmark, and naive Bayes provides a fast probabilistic baseline, but both rest on assumptions that appear only partially satisfied in this dataset.

My final submission package should include this notebook, the Kaggle screenshots, the short APA-style written report, and the GitHub repository link.